# MERGEN — Biyolojik Tabanlı Dijital Beyin Eğitim Arayüzü (V8.0)

Bu notebook, tüm Mergen deposunun Google Drive'a taşınarak Colab üzerinde çalıştırılması esasına göre tasarlanmıştır. Bu sayede kodlar, eğitim durumu ve ağırlık dosyaları doğrudan Google Drive üzerinde gerçek zamanlı olarak senkronize edilir.

**Temiz Eğitim Planı:** Bu arayüz, eğitim öncesinde tüm doğuştan gelen priors ağırlıklarını (`scripts/generate_innate_priors.py`) ve temel kelime dağarcığı sinaptik ağırlıklarını (`scripts/train_vocabulary.py`) Colab ortamında sıfırdan ve tertemiz yeniden oluşturur.

In [ ]:
# 1. Google Drive Bağlantısı
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Çalışma Dizini Ayarı
# Mergen deposunun Drive üzerindeki yolunu buraya yazın (Örnek: Mergen-main)
%cd /content/drive/MyDrive/Mergen-main/
!pwd

In [ ]:
# 3. Donanım ve GPU Kontrolü
!nvidia-smi

In [ ]:
# 4. Bağımlılıkların Yüklenmesi
# Tüm mimari paket gereksinimleri repodan yüklenir
!pip install -r requirements.txt

## 5. Temiz Kurulum: Priors ve Kelime Dağarcığı Ağırlıklarının Sıfırdan Üretilmesi

Eğitimin tamamen temiz bir sayfadan başlaması için priors ağırlıklarını ve kelime dağarcığı sinaptik bağlantılarını sıfırdan üretin.

In [ ]:
# 5.a Doğuştan Gelen (Innate) Priors Ağırlıklarını Sıfırdan Üret
!python scripts/generate_innate_priors.py

In [ ]:
# 5.b Kelime Dağarcığı Ağırlıklarını STDP ve Ödülle Sıfırdan Eğit
# Bu adım priors ağırlıklarını yükler ve %99+ başarıyla kelime bağlantılarını (mergen_weights.mx) oluşturur
!python scripts/train_vocabulary.py

## 6. Wikipedia Veri Hazırlığı (bz2 -> part_*.txt)

Aşağıdaki hücre, kök dizindeki `wikitr.bz2` dump dosyasını RAM sızdırmaz iterparse metoduyla tarayarak temiz Türkçe deneyimlere dönüştürür ve `data/full_circullum_training/` dizini altında 15.000'er satırlık partlar halinde yazar.

In [ ]:
# 6. Veri Hazırlama Scriptini Çalıştır
!python prepare_wiki_data.py --input ./wikitr.bz2 --output_dir ./data/full_circullum_training

## 7. Müfredatlı Otonom Eğitimi Başlat

Bu hücre sırasıyla şu adımları yürütür:
1. İlk olarak çekirdek müfredat dosyasını (`data/training/core_curriculum_v1.txt`) baştan sona eğitir, rüya döngüsünü çalıştırıp kaydeder.
2. Çekirdek müfredat tamamlandıktan sonra, `data/full_circullum_training/` dizinindeki Wikipedia partlarını sıralı olarak eğitmeye başlar.
3. Her part tamamlandığında `dream.sleep()` senkron konsolidasyon döngüsünü çalıştırır ve Drive'a checkpoint kaydeder.
4. Bağlantı koparsa, bu hücreyi yeniden çalıştırmanız yeterlidir; `colab_training_state.json` dosyasını okuyarak kaldığı dosyadan ve satırdan otonom olarak devam eder.

In [ ]:
# 7. Otonom Eğitim Döngüsü
!python scripts/train_colab.py \
    --corpus_dir "./data/full_circullum_training/" \
    --checkpoint_dir "./checkpoints/" \
    --core_path "./data/training/core_curriculum_v1.txt" \
    --mx_path "./mergen_weights.mx" \
    --vocab_path "./mergen_vocab.json" \
    --sleep_interval 500 \
    --sleep_cycles 1000 \
    --som_lr 0.05 \
    --stdp_reward 1.0 \
    --device cuda

## 8. Eğitim Durumu ve Canlı Telemetri Takibi

In [ ]:
# 8. Durum JSON Dosyasını Oku
import os
import json
state_path = "./checkpoints/colab_training_state.json"

if os.path.exists(state_path):
    with open(state_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print("=== MERGEN EĞİTİM DURUM RAPORU ===")
    print(f"Son Güncelleme Zamanı  : {data.get('last_update')}")
    print(f"Çekirdek Müfredat Durumu: {'TAMAMLANDI' if data.get('core_completed') else 'EĞİTİLİYOR'}")
    print(f"Eğitilen Toplam Cümle  : {data.get('total_sentences_trained'):,}")
    print(f"En Son İşlenen Dosya    : {data.get('last_processed_file')}")
    print(f"İşlenen Toplam Part Sayısı: {len(data.get('processed_chunks', []))}")
else:
    print("Eğitim henüz başlatılmadı veya checkpoints/ klasörü altında durum dosyası bulunamadı.")

## 9. Canlı Günlük (Log) Çıktılarını Görüntüle

In [ ]:
# 9. Son 50 satırı yazdırır (tarayıcı kapansa bile Drive logs/ klasöründen takip edebilirsiniz)
!tail -n 50 ./logs/training.log